# ZeroMall Fashion AI • FASHN VTON v1.5 Google Colab Demo (Miễn Phí 100%)

Notebook này giúp bạn chạy thử nghiệm mô hình mã nguồn mở SOTA **FASHN VTON v1.5** (Apache 2.0 License) trên GPU Nvidia T4 miễn phí của Google Colab.

### Hướng dẫn:
1. Vào menu **Runtime** > **Change runtime type** > Chọn **T4 GPU**.
2. Chạy lần lượt các ô lệnh bên dưới (hoặc bấm **Ctrl + F9** / **Runtime > Run all**).
3. Mở link Public Gradio để tải ảnh của bạn và ảnh trang phục lên thử đồ!

In [ ]:
# 1. Clone mã nguồn FASHN VTON v1.5 và cài đặt thư viện
!git clone https://github.com/fashn-AI/fashn-vton-1.5.git
%cd fashn-vton-1.5
!pip install -e .
!pip install gradio onnxruntime-gpu

In [ ]:
# 2. Tải trọng số mô hình AI (~2 GB) từ HuggingFace
!python scripts/download_weights.py --weights-dir ./weights

In [ ]:
# 3. Khởi tạo Web UI Gradio Thử Đồ Trực Quan
import gradio as gr
from PIL import Image
from fashn_vton import TryOnPipeline

# Khởi tạo Pipeline tối ưu trên GPU T4
pipeline = TryOnPipeline(weights_dir="./weights")

def predict_tryon(person_img, garment_img, category, timesteps):
    if person_img is None or garment_img is None:
        return None
    result = pipeline(
        person_image=person_img.convert("RGB"),
        garment_image=garment_img.convert("RGB"),
        category=category, # "tops", "bottoms", "one-pieces"
        num_timesteps=int(timesteps),
        garment_photo_type="flat-lay",
        segmentation_free=True
    )
    return result.images[0]

demo = gr.Interface(
    fn=predict_tryon,
    inputs=[
        gr.Image(type="pil", label="1. Ảnh Toàn Thân Người Mẫu / Của Bạn"),
        gr.Image(type="pil", label="2. Ảnh Sản Phẩm Trang Phục (Áo/Quần ZeroMall)"),
        gr.Radio(["tops", "bottoms", "one-pieces"], value="bottoms", label="3. Danh Mục Trang Phục"),
        gr.Slider(minimum=15, maximum=40, value=25, step=5, label="4. Số Bước Denoising (15: Nhanh, 25: Cân Bằng, 35: Siêu Nét)")
    ],
    outputs=gr.Image(label="✨ Kết Quả Mặc Thử Thực Tế"),
    title="🛍️ ZeroMall Virtual Try-On AI • Powered by FASHN VTON v1.5",
    description="Trực tiếp sinh ảnh người mặc thử sản phẩm trên không gian Pixel Space (MMDiT Transformer), không phụ thuộc Agnostic Mask."
)

# Tạo đường link public dùng được trên mọi thiết bị
demo.launch(share=True)